# Notebook 34 — Corrected V4.3 Warm-Retention Calibration

This validation-only experiment does **not** retrain the Transformer and does **not** read Test. It keeps the fixed Corrected V4 author/popularity policy, then adds a static/long-term score from the same checkpoint to recover warm-item strength.

All weights are selected on calibration users and evaluated once on disjoint validation holdout users. A branch advances only if overall and warm HR@1000 have positive paired confidence intervals while cold HR@1000 does not regress. Legacy V4.2 is a reference gate, not a valid online-state contract.

Use an **A100 or L4 GPU**. Exact matrix scoring covers the complete validation catalog; no model training occurs.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')

from getpass import getpass
from pathlib import Path
import hashlib, importlib.util, json, shutil, subprocess, sys, zipfile

DRIVE_ROOT = Path('/content/drive/MyDrive/recsys')
ADAPTIVE_ROOT = DRIVE_ROOT / 'data/gold/adaptive/v1'
BASE_GOLD_ROOT = DRIVE_ROOT / 'data/gold/two_tower/v1_colab'
PROTOCOL_DIR = ADAPTIVE_ROOT / 'evaluation_protocol/v2'
WORK_ROOT = Path('/content/recsys_v4_3_warm_retention')
SOURCE_ROOT = WORK_ROOT / 'runtime'
HF_SNAPSHOT_ROOT = WORK_ROOT / 'hf_snapshot'
CORRECTED_ROOT = HF_SNAPSHOT_ROOT / 'corrected_sequences/v2'
CORRECTED_V4_DIR = WORK_ROOT / 'artifacts/corrected_v4'
LEGACY_V4_DIR = WORK_ROOT / 'artifacts/legacy_v4'
REPORT_DIR = WORK_ROOT / 'reports/v4_3_warm_retention'
DRIVE_REPORT_DIR = DRIVE_ROOT / 'artifacts/adaptive/v4_3_warm_retention_v1'
for path in (SOURCE_ROOT, CORRECTED_V4_DIR, LEGACY_V4_DIR, REPORT_DIR, DRIVE_REPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)
print('work root:', WORK_ROOT)

In [ ]:
required = {'torch': 'torch', 'pandas': 'pandas', 'pyarrow': 'pyarrow', 'huggingface_hub': 'huggingface_hub>=0.32'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import torch
assert torch.cuda.is_available(), 'Select an A100 or L4 GPU runtime.'
free_gib = shutil.disk_usage('/content').free / 1024**3
assert free_gib >= 15, f'At least 15 GiB local disk is required; only {free_gib:.1f} GiB is free.'
print('GPU:', torch.cuda.get_device_name(0))
print(f'local disk free: {free_gib:.1f} GiB')

In [ ]:
required_paths = [
    ADAPTIVE_ROOT / 'sequence_examples/train',
    ADAPTIVE_ROOT / 'sequence_examples/validation',
    BASE_GOLD_ROOT / 'item_features/static',
    BASE_GOLD_ROOT / 'transforms/numeric_stats.json',
    BASE_GOLD_ROOT / 'vocabularies/user_active_degree',
    BASE_GOLD_ROOT / 'vocabularies/video_type',
    BASE_GOLD_ROOT / 'vocabularies/upload_type',
    PROTOCOL_DIR / 'queries.parquet',
    PROTOCOL_DIR / 'manifest.json',
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, 'Missing Drive inputs:\n' + '\n'.join(missing_paths)
protocol_manifest = json.loads((PROTOCOL_DIR / 'manifest.json').read_text())
assert protocol_manifest['builder_revision'] == '2.1-session-history'
print('Drive/A2 validation preflight: PASS')
print('queries/users:', protocol_manifest['queries'], protocol_manifest['users'])

## Download verified runtime and frozen artifacts

Enable the existing `HF_TOKEN` Colab Secret. No write token is needed.

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download

HF_REPO_ID = 'khoa1505/recsys-artifacts'
RUNTIME_FILE = 'runtime/corrected_v4_warm_retention_runtime_v1.zip'
RUNTIME_SHA256 = '4556e9c50fe7688957cd44d29b55e7562e82555c82dc45567d7b308af78b2937'
CORRECTED_V4_FILE = 'v4_corrected/v1/best_model.pt'
LEGACY_V4_FILE = 'v4/v4_inference_fp16.pt'
LEGACY_V4_SHA256 = '4ad49b3e6f9f8b5d45a6c92f31d8d9b27f558e1746613efb44a23db3cd456a26'

def normalize_token(raw):
    if isinstance(raw, bytes): raw = raw.decode()
    if isinstance(raw, str):
        raw = raw.strip()
        return raw if raw.startswith('hf_') else None
    if isinstance(raw, dict):
        for value in raw.values():
            token = normalize_token(value)
            if token: return token
    return None

try:
    HF_TOKEN = normalize_token(userdata.get('HF_TOKEN'))
except Exception:
    HF_TOKEN = None
HF_TOKEN = HF_TOKEN or normalize_token(getpass('Paste Hugging Face READ token: '))
assert HF_TOKEN, 'HF_TOKEN is missing or invalid.'

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        while block := stream.read(8 * 1024 * 1024): digest.update(block)
    return digest.hexdigest()

runtime_zip = Path(hf_hub_download(HF_REPO_ID, RUNTIME_FILE, repo_type='model', token=HF_TOKEN))
corrected_v4 = Path(hf_hub_download(HF_REPO_ID, CORRECTED_V4_FILE, repo_type='model', token=HF_TOKEN))
legacy_v4 = Path(hf_hub_download(HF_REPO_ID, LEGACY_V4_FILE, repo_type='model', token=HF_TOKEN))
assert sha256(runtime_zip) == RUNTIME_SHA256, 'Runtime ZIP hash mismatch'
assert sha256(legacy_v4) == LEGACY_V4_SHA256, 'Legacy V4 checkpoint hash mismatch'
assert corrected_v4.stat().st_size > 100_000_000, 'Corrected V4 checkpoint is unexpectedly small'
snapshot_download(HF_REPO_ID, repo_type='model', token=HF_TOKEN, local_dir=HF_SNAPSHOT_ROOT, allow_patterns=['corrected_sequences/v2/manifest.json', 'corrected_sequences/v2/sequences/split=validation/**'])
corrected_manifest = json.loads((CORRECTED_ROOT / 'manifest.json').read_text())
assert corrected_manifest['quality']['passed'] is True
for source, destination in ((corrected_v4, CORRECTED_V4_DIR / 'best_model.pt'), (legacy_v4, LEGACY_V4_DIR / 'best_model.pt')):
    destination.unlink(missing_ok=True)
    destination.symlink_to(source.resolve())
print('runtime/checkpoints/corrected validation delta: PASS')

In [ ]:
if SOURCE_ROOT.exists(): shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(runtime_zip) as archive:
    assert archive.testzip() is None, 'Runtime archive is corrupt'
    archive.extractall(SOURCE_ROOT)
EVAL_SCRIPT = SOURCE_ROOT / 'scripts/evaluate_corrected_v4_warm_retention.py'
assert EVAL_SCRIPT.exists(), EVAL_SCRIPT
compile(EVAL_SCRIPT.read_text(), str(EVAL_SCRIPT), 'exec')
print('runtime extraction/compile: PASS')

## Calibrate on validation users and evaluate disjoint holdout users

The grid contains one zero-weight baseline plus positive static weights in global and warm-only modes. Test artifacts are never loaded.

In [ ]:
command = [
    sys.executable, str(EVAL_SCRIPT),
    '--adaptive-root', str(ADAPTIVE_ROOT), '--base-gold-root', str(BASE_GOLD_ROOT),
    '--corrected-root', str(CORRECTED_ROOT), '--corrected-v4-dir', str(CORRECTED_V4_DIR),
    '--legacy-v4-dir', str(LEGACY_V4_DIR), '--protocol-dir', str(PROTOCOL_DIR),
    '--output-dir', str(REPORT_DIR),
    '--static-weights', '0,0.05,0.1,0.2,0.3,0.5,0.75,1.0', '--static-modes', 'all,warm',
    '--author-weight', '0.2', '--popularity-weight', '0.02', '--recency-tau', '3.0',
    '--calibration-user-fraction', '0.35', '--query-batch-size', '16', '--item-batch-size', '8192',
    '--bootstrap-samples', '2000', '--seed', '44', '--device', 'cuda',
]
eval_log = Path('/content/corrected_v4_3_warm_retention.log')
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with eval_log.open('w') as log:
    for line in process.stdout:
        print(line, end=''); log.write(line); log.flush()
code = process.wait()
if code != 0:
    tail = '\n'.join(eval_log.read_text(errors='replace').splitlines()[-240:])
    raise RuntimeError(f'V4.3 validation failed with exit code {code}:\n{tail}')
REPORT_JSON = REPORT_DIR / 'corrected_v4_3_warm_retention.json'
assert REPORT_JSON.exists(), REPORT_JSON
print('V4.3 validation evaluation: PASS')

In [ ]:
import pandas as pd
report = json.loads(REPORT_JSON.read_text())
print('selected:', report['selected'])
print('gates:', json.dumps(report['acceptance_gates'], indent=2))
display(pd.read_csv(REPORT_DIR / 'holdout_summary.csv'))
display(pd.read_csv(REPORT_DIR / 'holdout_segments.csv').query("segment_type == 'target_warm'"))
print('overall HR@1000 delta:', report['paired_user_bootstrap']['selected_vs_corrected_base']['hitrate@1000'])
print('warm HR@1000 delta:', report['paired_user_bootstrap']['selected_vs_base_warm']['hitrate@1000'])

In [ ]:
small_report = WORK_ROOT / 'small_report'
shutil.rmtree(small_report, ignore_errors=True); small_report.mkdir()
for name in ('corrected_v4_3_warm_retention.json', 'calibration_grid.csv', 'holdout_summary.csv', 'holdout_segments.csv'):
    source = REPORT_DIR / name
    assert source.exists(), source
    shutil.copy2(source, small_report / name)
shutil.copy2(eval_log, small_report / eval_log.name)
for path in small_report.iterdir(): shutil.copy2(path, DRIVE_REPORT_DIR / path.name)
archive = Path(shutil.make_archive('/content/corrected_v4_3_warm_retention_report', 'zip', small_report))
print('Drive report:', DRIVE_REPORT_DIR)
print('download:', archive, f'{archive.stat().st_size / 1024**2:.2f} MiB')
files.download(str(archive))
print('VALIDATION DECISION:', 'ADVANCE' if report['accepted'] else 'STOP')
print('PRODUCTION PROMOTION:', report['acceptance_gates']['production_promotion'])

## Decision semantics

- `accepted_for_next_stage=true`: the static branch improves overall and warm HR@1000 on disjoint validation users without lowering cold HR@1000.
- `production_promotion=true`: it additionally beats Legacy V4.2 with a positive paired interval.
- If validation returns `STOP`, do not run another locked-Test evaluation. Move directly to richer semantic item features or an ALS warm-candidate source.
- Test remains sealed; this notebook cannot silently tune on the previously consumed test report.